# ROS 2 control cycle timing

Set `ZFC_DATA_DIR` to a directory of ROS 2 runs, each with `manifest.json` and
`timing.bin`. `profiling/fixtures` contains only artificial examples.
Times are integer nanoseconds; no clock-cost subtraction is applied.
The exclusive Controller Manager stages cover cycle entry through sleep entry.
Nested EtherCAT stages must not be added to their parent hardware spans.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT/'packages/zfc_ethercat_core').is_dir():
    if ROOT == ROOT.parent: raise RuntimeError('Run from inside the repository')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'profiling/analysis'))
from trace import load_run, summary, cm_partition
DATA = Path(os.environ.get('ZFC_DATA_DIR', ROOT/'profiling/results'))
OUT = Path(os.environ.get('ZFC_PLOT_DIR', ROOT/'profiling/results/plots'))
OUT.mkdir(parents=True, exist_ok=True)
runs = {}
for manifest in sorted(DATA.glob('*/manifest.json')):
    if (manifest.parent/'timing.bin').exists(): runs[manifest.parent.name] = load_run(manifest.parent)
if not runs: raise RuntimeError(f'No timing runs in {DATA}; use ZFC_DATA_DIR=profiling/fixtures for a synthetic test')
print('SYNTHETIC DATA ONLY — no hardware feasibility conclusion' if all(m['synthetic'] for d,m,i in runs.values()) else 'Inspect manifest conditions and phase coverage for each run')
display(pd.DataFrame([dict(run=k, **m, **i) for k,(d,m,i) in runs.items()]).drop(columns=['environment']))


## Integrity and tails

Missing cycles and final trace drops are reported separately; a final buffer overflow can lose the tail
without leaving holes inside the retained records. Unknown deadlines remain missing. p99.9 is shown only
at n ≥ 1,000 and p99.99 at n ≥ 10,000; these thresholds permit computing the rank, not precise tail inference.
CM deadlines are mapped from its system-clock schedule to monotonic time using a bracketed clock pair;
inspect mapping-window error and clock discontinuities before interpreting deadline misses.

In [ ]:
metrics = ['actual_period_ns','period_error_ns','wakeup_lateness_ns','active_span_ns','execution_ns','budget_slack_ns','completion_lateness_ns','hardware_read_ns','core_read_ns','read_adapter_ns','controller_ns','calculation_ns','controller_wrapper_ns','hardware_write_ns','core_write_ns','write_adapter_ns','profile_commit_ns']
rows=[]
for name,(d,m,integrity) in runs.items():
    for phase,g in d.groupby('phase'):
        for metric in metrics:
            rows.append(dict(run=name,phase=phase,metric=metric,**summary(g[metric])))
    known=d.deadline_miss.dropna()
    print(name, 'cycles',len(d),'missing',integrity['missing_cycles'],'trace drops',integrity['trace_drops'],
          'diagnostic drops',int(d.diagnostic_drops.max()), 'fault records',int(d.fault.ne(0).sum()),
          'execution overruns',int(d.execution_overrun.fillna(False).sum()), 'known deadlines',len(known),'misses',int(known.sum()),'miss rate',known.mean(),
          'maximum mapping window ns',int(d.clock_mapping_window_ns.max()))
stats=pd.DataFrame(rows)
display(stats)
stats.to_csv(OUT/'summary.csv',index=False)


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
for name,(d,m,i) in runs.items():
    x=np.sort(d.period_error_ns.dropna().to_numpy())
    axes[0].hist(x,bins=60,histtype='step',density=True,label=name)
    axes[1].plot(x,np.arange(1,len(x)+1)/len(x),label=name)
    tail=np.sort(np.abs(x)); positive=tail>0
    axes[2].loglog(tail[positive],(len(tail)-np.arange(len(tail)))[positive]/len(tail),label=name)
for ax,title in zip(axes,['Period error histogram','Period error ECDF','Absolute period error survival']):
    ax.set_title(title); ax.set_xlabel('ns'); ax.legend()
fig.tight_layout();fig.savefig(OUT/'period-tails.png');plt.show()
fig,axes=plt.subplots(1,2,figsize=(12,4))
for name,(d,m,i) in runs.items():
    for ax,metric in zip(axes,['execution_ns','wakeup_lateness_ns']):
        x=np.sort(d[metric].dropna().to_numpy()); positive=x>0
        ax.loglog(x[positive],(len(x)-np.arange(len(x)))[positive]/max(len(x),1),label=name)
        ax.set_title(metric);ax.set_xlabel('ns');ax.set_ylabel('P(X ≥ x)');ax.legend()
fig.tight_layout();fig.savefig(OUT/'execution-tails.png');plt.show()


In [ ]:
for name,(d,m,i) in runs.items():
    fig,axes=plt.subplots(2,1,figsize=(12,6))
    t=(d.cycle_entry-d.cycle_entry.iloc[0])/1e9
    for ax,metric in zip(axes,['period_error_ns','execution_ns']):
        ax.plot(t,d[metric],alpha=.5,label=metric)
        ax.plot(t,d[metric].rolling(min(1000,max(1,len(d)//4)),min_periods=1).max(),label='rolling maximum')
        ax.set_ylabel('ns');ax.legend()
    axes[1].set_xlabel('elapsed seconds');fig.suptitle(name)
    fig.tight_layout();fig.savefig(OUT/f'{name}-series.png');plt.show()


## Mutually exclusive framework stages

These stages close algebraically to cycle entry → sleep entry. They are available only for CM cycles
with one controller update and one hardware read/write. Missing boundaries are not zero overhead.
Execution time covers cycle entry → sleep entry; active span covers hardware read entry → hardware write exit.
Mean stacks use means of per-cycle stages, never sums of separately computed percentiles.

In [ ]:
partitions={name:cm_partition(d).dropna() for name,(d,m,i) in runs.items()}
partitions={name:p for name,p in partitions.items() if len(p)}
for name,p in partitions.items():
    display(pd.DataFrame({k:summary(p[k]) for k in p}).T)
    fig,axes=plt.subplots(1,3,figsize=(18,5))
    p.plot.box(ax=axes[0],rot=90);axes[0].set_ylabel('ns');axes[0].set_title(name+' stage distributions')
    p.plot(ax=axes[1],legend=False);axes[1].set_ylabel('ns / cycle')
    (p/10000).plot(ax=axes[2]);axes[2].set_ylabel('% of 1 ms budget / cycle')
    fig.tight_layout();fig.savefig(OUT/f'{name}-stages.png');plt.show()
if partitions:
    means=pd.DataFrame({name:p.mean() for name,p in partitions.items()}).T
    ax=means.plot.bar(stacked=True,figsize=(12,6));ax.set_ylabel('mean ns');ax.set_title('Exclusive mean budget composition')
    plt.tight_layout();plt.savefig(OUT/'mean-budget.png');plt.show()
    display(means)


## Nested EtherCAT and adapter costs

Receive/send API times include application/kernel API work; they are not Ethernet wire-transit times.
FINE is a separate perturbation trial. Core totals also include probe work and state copies outside
the individually timed groups. Adapter remainder is body minus core; diagnostic construction/push are
nested inside the hardware body. The commit column measures the append copy and surrounding two clocks;
the complete profiler also includes record construction and every probe.

In [ ]:
fine=['application_time_ns','receive_api_ns','domain_process_ns','elm_decode_ns','motor_decode_ns','state_poll_ns','readiness_ns','encode_ns','reference_sync_ns','slave_sync_ns','domain_queue_ns','send_api_ns','diagnostic_construct_ns','diagnostic_push_ns']
for name,(d,m,i) in runs.items():
    if i['level']!=2: continue
    display(pd.DataFrame({k:summary(d[k]) for k in fine}).T)
    ax=d[fine].plot.box(rot=90,figsize=(12,6));ax.set_ylabel('ns');ax.set_title(name+' FINE nested costs')
    plt.tight_layout();plt.savefig(OUT/f'{name}-fine.png');plt.show()


In [ ]:
for name,(d,m,i) in runs.items():
    factors=['receive_api_ns','send_api_ns','reference_sync','diagnostic_construct_ns','diagnostic_push_ns','command_change','cpu_migration','wc','wc_state','ready','fault']
    table=d[['period_error_ns','execution_ns']+factors].corr(method='spearman')
    display(table.loc[factors,['period_error_ns','execution_ns']].rename_axis(name))
    print('CPU-change observations are a lower bound on migrations; within-cycle out-and-back migration is not observable from one CPU sample.')


## Interpretation

Report integrity, phase coverage, sample counts, period error, cycle execution,
slack, overruns and mapped deadline misses. Inspect clock mapping uncertainty.
No-motion runs do not characterize changed-command diagnostics during motion.
Finite captures give maximum observed values, not WCET or hard real-time proof.
Synthetic data validates analysis only. Internal Resource Manager mutex waiting
cannot be separated from other wrapper work by these probes.
